# MedCLIPSeg++ — scientific rationale and benchmarking

This notebook explains why the study chose its model, checkpoint, uncertainty operators, thresholds, metrics, and evaluation design. It presents reported evidence with its limitations; it does not independently recompute study CSVs.

## Six-stage system
| Stage | Role | Research rationale |
|---|---|---|
| 1 Inputs | Image and request | Define scan and prompt |
| 2 Intent router | Choose segmentation/retrieval | Route both operations |
| 3 Segmentation | Frozen MedCLIPSeg outputs mask and pixelwise entropy | Isolate post-hoc selection from retraining |
| 4 Selective prediction | Aggregate uncertainty and accept/defer | Test a training-free quality-ranking score |
| 5 Similarity retrieval | Metadata-filtered vector top-K with PABP | Separate systems contribution |
| 6 System output | Present mask, uncertainty, triage, neighbors | Review with provenance |

Questions: which of seven aggregators ranks quality; how a validation-frozen cutoff transfers; how four retrieval strategies compare on a controlled workload; and which claims are supported or falsified.

> Scope: post-hoc segmentation-quality selection, not a new trained model, a guarantee that confidence equals accuracy, or clinical validation. The PABP harness uses generated random vectors and synthetic eligibility labels, not clinical embeddings.

**Canonical sources:** [Experimental Protocol](../uncertainty_rejection/research/EXPERIMENTAL_PROTOCOL.md) · [Published Baseline Comparison](../uncertainty_rejection/research/PUBLISHED_BASELINE_COMPARISON.md) · [Research Decisions and Limitations](../uncertainty_rejection/research/RESEARCH_DECISIONS_AND_LIMITATIONS.md) · [Scientific Evidence and Claims](../uncertainty_rejection/research/SCIENTIFIC_EVIDENCE_AND_CLAIMS.md) · [Theory and Metrics](../uncertainty_rejection/research/THEORY_AND_METRICS.md)


## 1. Why this model, checkpoint, and protocol?

| Choice | Rationale | Fixed or unresolved |
|---|---|---|
| **MedCLIPSeg, UniMedCLIP ViT-B/16, PVL adapters** | Published probabilistic vision-language segmentation; stochastic output exposes pixel entropy for post-hoc study without a learned quality model | Frozen inference; no weights retrained |
| **Authors' released checkpoint lineage** | Baseline report states near-parity with published CVPR results on BUSI and four external breast-US cohorts | Project path is recorded, but checkpoint hash and original training lineage still require release-level verification |
| **T=30, temperature 0.20, entropy grid 224×224, seed 666** | Reported inference settings produce mean prediction and raw entropy | Protocol settings, not notebook-tuned; entropy grid differs from evaluation-mask space |
| **BUSI + UDIAT, BUID, BUSUC, BUSBRA** | Tests transfer while keeping the organ consistent | Scanner, collection, annotation factors are not isolated; image IDs do not establish patient independence |
| **Validation → locked test → external test** | Select score and cutoff on validation, then freeze to observe transfer and coverage drift | No external recalibration; historical exploratory test inspection is disclosed |

This is reported execution equivalence, not de novo model training or independent reproduction. Checkpoint hash and training lineage remain release-provenance tasks.


In [ ]:
# Cell 1: Environment Setup & Dependencies
import sys
import os
import time
from pathlib import Path
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from PIL import Image

# Configure Matplotlib
import matplotlib
matplotlib.use('Agg')

# Resolve repository root
repo_root = Path.cwd().resolve()
if not (repo_root / "stages").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from stages import (
    BaseSegmentationModelAdapter,
    MockModelAdapter,
    compute_all_uncertainty_metrics,
    evaluate_decision_policy,
    FROZEN_TAU_STAR,
    FROZEN_TAU_NEG,
    PABPRetrievalEngine,
    benchmark_vector_search_algorithms,
    VectorSearchBenchmarkSuite,
)

print(f"[OK] Scientific benchmarking environment ready. Repo root: {repo_root}")


## 2. Why compare seven uncertainty operators?

For each image, the model produces a dense predictive entropy map H and a predicted binary foreground F. Entropy is an uncertainty summary, not a pixelwise error label. Each operator maps H to one image-level score; comparison uses the same 65 lesion-positive BUSI validation cases. Primary loss is 1−DSC; secondary loss is 1−NSD. Smaller AURC means lower empirical loss among retained cases over the evaluated coverage curve.

| Operator | Definition | Why include it | Val AURC DSC / NSD | Decision |
|---|---|---|---:|---|
| **U_fg** | Mean H over predicted foreground F | Scores ambiguity within predicted lesion | **0.0839 / 0.0563** | Selected primary; simple, lowest point AURC, lower observed area association |
| U_std_fg | Std(H over F) | Tests added value of spatial dispersion | 0.0850 / 0.0573 | Close comparator; paired difference unresolved (p=0.294) |
| U_boundary | Mean H over morphological band B | Tests contour-focused uncertainty | 0.1302 / 0.0848 | Surface comparator; weaker for region DSC ranking |
| U_ratio | U_fg / (U_mean + 10⁻⁶) | Tests foreground-to-global contrast | 0.1368 / 0.1159 | Included; potential area/scale sensitivity |
| U_mean | Mean H over all pixels Ω | Global whole-image baseline | 0.1438 / 0.0969 | Predicted-area rho +0.9148 in validation |
| U_positive | Mean H where H>0 | Tests ignoring exact-zero entropy pixels | 0.1455 / 0.0984 | Did not improve DSC ranking |
| U_threshold | Mean H where H>0.1 | Tests conditional high-entropy aggregation | 0.1750 / 0.1412 | Worst validation AURC here |

**Why choose U_fg over U_std_fg?** U_fg's point estimate is lower, but paired ΔAURC is −0.0011 (95% CI [−0.0033,+0.0008], p=0.294); superiority is unresolved. It was selected as a simple, interpretable primary score with lower observed area association than global averaging, not as a proven size-independent or novel formula. Empty regions are undefined, not zero uncertainty.

The study's boundary B is derived by morphological dilation/erosion with a square 5×5 kernel; it is not a physical 2-mm contour. The empirical comparison uses the predicted mask, never ground-truth pixels, to calculate inference-time scores. Ground truth enters only evaluation metrics and oracle reference.


### Synthetic mechanism illustration — not empirical BUSI evidence

The following cell constructs circles and a hand-designed entropy field to visualize how area and aggregation can interact in that simulation. Its correlations are not study results and cannot prove U_fg is independent of lesion size.


In [ ]:
# Synthetic mechanism illustration only; this is not an empirical BUSI experiment.
# It demonstrates how the chosen simulated entropy field behaves as lesion area changes.
# Cell 3: Quantitative Comparison of Uncertainty Operators Across Lesion Sizes
np.random.seed(666)

# Generate synthetic lesions across varying sizes (50 to 5000 pixels) to test area confounding
simulated_areas = np.logspace(1.7, 3.7, 50) # 50 to ~5000 pixels
operator_results = []

for area in simulated_areas:
    h, w = 256, 256
    radius = np.sqrt(area / np.pi)
    y, x = np.ogrid[:h, :w]
    mask = ((y - 128)**2 + (x - 128)**2 <= radius**2).astype(np.uint8)
    
    # Boundary-concentrated predictive entropy (independent of lesion size)
    dist = np.abs(np.sqrt((y - 128)**2 + (x - 128)**2) - radius)
    entropy = np.exp(-dist * 2.5) * 0.25
    entropy += np.random.uniform(0.005, 0.02, size=(h, w)) # background acoustic noise
    
    metrics = compute_all_uncertainty_metrics(entropy, mask)
    metrics["true_area"] = float(np.sum(mask > 0))
    operator_results.append(metrics)

df_ops = pd.DataFrame(operator_results)

# Compute Spearman rank correlation with lesion footprint
corr_ufg, _ = spearmanr(df_ops["u_fg"], df_ops["true_area"])
corr_umean, _ = spearmanr(df_ops["u_mean"], df_ops["true_area"])
corr_upos, _ = spearmanr(df_ops["u_positive"], df_ops["true_area"])
corr_ubnd, _ = spearmanr(df_ops["u_boundary"], df_ops["true_area"])

print("="*75)
print(" SPEARMAN CORRELATION WITH LESION FOOTPRINT (AREA CONFOUNDING ANALYSIS)")
print("="*75)
print(f"  • Global Grid Mean (U_mean)     : rho = {corr_umean:+.4f} (EXTREME AREA CONFOUNDING)")
print(f"  • Non-Zero Positive (U_positive): rho = {corr_upos:+.4f} (Moderate Confounding)")
print(f"  • Boundary Band (U_boundary)    : rho = {corr_ubnd:+.4f} (Low Confounding)")
print(f"  • Foreground Stencil (U_fg)     : rho = {corr_ufg:+.4f} (DECOUPLED FROM AREA)")
print("="*75)


### Cohort design and edge cases

| Cohort | N | Role |
|---|---:|---|
| BUSI validation | 78 total; 65 lesion-positive, 13 normal | Only cohort for method selection and cutoff calibration; AURC comparison uses the 65 lesion cases |
| BUSI locked test | 78 total; 65 lesion-positive, 13 normal | In-domain evaluation after method and cutoff freeze |
| UDIAT, BUID, BUSUC, BUSBRA | 25, 35, 122, 282 | Four external breast-US test cohorts, total N=464; no threshold recalibration |

On BUSI, normal empty/empty predictions make several foreground or boundary scores undefined. They are not assigned zero uncertainty for the seven-method lesion-ranking comparison. The two-stage policy handles a predicted-empty mask separately with U_mean and τ_neg; this branch is not established as a true-negative safeguard. Patient-level independence is not guaranteed by non-overlapping image IDs.



## 3. Why these metrics and thresholds?

| Quantity | Purpose | Interpretation |
|---|---|---|
| DSC, loss 1−DSC | Primary region endpoint | Higher DSC is better |
| NSD, loss 1−NSD | Secondary surface endpoint | Tolerance reported in pixels, not verified millimeters |
| Spearman rho | Relates score ranking to observed quality | Descriptive association, not causality or calibration |
| AURC | Risk over retained coverage | Empirical quality/coverage comparison on matched cases |
| Paired bootstrap CI | Same-case AURC contrast uncertainty | Tiny unresolved differences are not wins |

**Why freeze τ*=0.064058324?** Chosen on BUSI validation lesion cases only to target 80% coverage (52/65), then frozen on locked test and OOD. This tests transfer without test-driven tuning. Realized coverage was 72.3% on BUSI test lesions and 51.3% pooled OOD.

The separate τ_neg=10⁻⁴ branch assigns a decision when the predicted mask is empty and U_fg is undefined. It does not confirm a true negative and is unvalidated as a missed-lesion safeguard. Other distinct settings: mask threshold 0.5, U_threshold pixel threshold 0.1, boundary width 2 (square 5×5), U_ratio epsilon 10⁻⁶, T=30.

**Counterexample:** BUSI test case 116_malignant.png had DSC 0.2193 but passed the frozen U_fg threshold. Perfect failure detection is contradicted.


## 4. Reported locked-test and external results

Values below are a snapshot transcribed from the protocol, not recomputed from per-image CSVs here. Pooled case-level AURC is a separate estimand, not the mean of the four external AURCs.


In [ ]:
MASTER_EVIDENCE = [
    {"Cohort":"BUSI locked test (lesion)","N":65,"U_fg_AURC":0.0790,"U_mean_AURC":0.1262,"Random_AURC":0.1687,"Oracle_AURC":0.0692,"Baseline_DSC":0.8286,"Retained_DSC":0.9031,"Coverage":"72.3%"},
    {"Cohort":"UDIAT","N":25,"U_fg_AURC":0.1133,"U_mean_AURC":0.1503,"Random_AURC":0.2432,"Oracle_AURC":0.0944,"Baseline_DSC":0.7467,"Retained_DSC":0.8747,"Coverage":"56.0%"},
    {"Cohort":"BUID","N":35,"U_fg_AURC":0.1094,"U_mean_AURC":0.2411,"Random_AURC":0.2040,"Oracle_AURC":0.0935,"Baseline_DSC":0.7900,"Retained_DSC":0.8818,"Coverage":"57.1%"},
    {"Cohort":"BUSUC","N":122,"U_fg_AURC":0.0916,"U_mean_AURC":0.1365,"Random_AURC":0.1557,"Oracle_AURC":0.0834,"Baseline_DSC":0.8430,"Retained_DSC":0.8982,"Coverage":"63.9%"},
    {"Cohort":"BUSBRA","N":282,"U_fg_AURC":0.1529,"U_mean_AURC":0.2020,"Random_AURC":0.2480,"Oracle_AURC":0.1200,"Baseline_DSC":0.7511,"Retained_DSC":0.8536,"Coverage":"44.7%"},
    {"Cohort":"Pooled external (case-level)","N":464,"U_fg_AURC":0.1267,"U_mean_AURC":0.1864,"Random_AURC":0.2216,"Oracle_AURC":0.1046,"Baseline_DSC":0.7780,"Retained_DSC":0.8718,"Coverage":"51.3%"},
]
df_master=pd.DataFrame(MASTER_EVIDENCE)
print(df_master.to_string(index=False))
print("\nRetained DSC is a selected-subset mean; selection does not change predicted masks.")
print("Coverage fell below the validation target on locked test and external cohorts.")


## 5. Why benchmark PABP?

The systems question is whether metadata-aware block pruning reduces vector comparisons while preserving top-K results. Four approaches share one workload: exact pre-filter scan (ground-truth reference), post-filter ANN, IVF with bitmap, and PABP. Metrics include Recall@10, p50/p95 latency, completeness, distance computations, and index build time.

PABP is evaluated because gallery queries can include eligibility predicates (for example, review status or cohort): its block summaries can skip blocks containing no eligible records. Exact pre-filter scan gives the eligible top-K reference; post-filter ANN tests recall loss from filtering after retrieval; IVF+bitmap checks metadata during inverted-list search. The harness generates random normalized vectors and synthetic eligibility labels (N=5,000, dimension 512, 40 queries per selectivity). This is a controlled software workload, not the clinical gallery. Results do not establish performance on real embeddings or a production catalog.


In [ ]:
# Cell 6: Run Comprehensive Vector Search Benchmark Suite
print("[INFO] Synthetic random-vector software benchmark; not clinical gallery embeddings.")

suite = VectorSearchBenchmarkSuite(embedding_dim=512, seed=42)

# Run benchmark across different selectivities (50%, 10%, 1%)
bench_results = suite.run_benchmark(
    n_vectors=5000,
    selectivities=[0.50, 0.10, 0.01],
    k=10,
    num_queries=40,
)

df_bench = pd.DataFrame([
    {
        "Algorithm": r.algorithm_name,
        "Selectivity": f"{int(r.selectivity * 100)}%",
        "Recall@10": f"{r.recall_at_k:.3f}",
        "Latency p50 (ms)": f"{r.latency_p50_ms:.3f}",
        "Latency p95 (ms)": f"{r.latency_p95_ms:.3f}",
        "QPS": f"{r.qps:.1f}",
        "Completeness": f"{r.result_completeness * 100:.1f}%",
        "Dot Products": int(r.avg_distance_computations),
        "Build (s)": f"{r.index_build_time_sec:.3f}",
    }
    for r in bench_results
])

print("="*105)
print(" VECTOR SEARCH BENCHMARK RESULTS (N=5,000 vectors, Dim=512, K=10)")
print("="*105)
print(df_bench.to_string(index=False))
print("="*105)


### Measurements from this benchmark run

The plots use the benchmark values computed above. They characterize only the synthetic workload and runtime in this session.


In [ ]:
bench_plot=pd.DataFrame([
    {"algorithm":r.algorithm_name,"selectivity":r.selectivity,"recall_at_k":r.recall_at_k,
     "latency_p50_ms":r.latency_p50_ms,"avg_distance_computations":r.avg_distance_computations}
    for r in bench_results
])
fig,axes=plt.subplots(1,3,figsize=(17,4.8))
for name,group in bench_plot.groupby("algorithm"):
    group=group.sort_values("selectivity")
    axes[0].plot(group["selectivity"],group["latency_p50_ms"],marker="o",label=name)
    axes[1].plot(group["selectivity"],group["recall_at_k"],marker="o",label=name)
    axes[2].plot(group["selectivity"],group["avg_distance_computations"],marker="o",label=name)
axes[0].set(title="Synthetic workload: p50 latency",xlabel="Eligibility selectivity",ylabel="Milliseconds")
axes[1].set(title="Synthetic workload: Recall@10",xlabel="Eligibility selectivity",ylabel="Recall")
axes[2].set(title="Synthetic workload: dot products",xlabel="Eligibility selectivity",ylabel="Mean computations")
for ax in axes:
    ax.grid(True,alpha=0.3)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()


## 6. Conclusions and limits

1. **Model:** the frozen published MedCLIPSeg baseline isolates post-hoc selection from retraining; checkpoint hash and training lineage remain provenance tasks.
2. **Aggregator:** U_fg had the lowest validation point AURC of seven operators, but difference from U_std_fg was not statistically resolved. It is pragmatic, not universally superior or novel.
3. **Threshold:** τ*=0.064058324 was selected on validation at target 80% and held fixed; coverage fell on locked test and external data.
4. **Failures:** a severe BUSI error passed the threshold; empty-mask fallback is not established as a true-negative detector. Uncertainty does not guarantee segmentation accuracy.
5. **Retrieval:** PABP uses synthetic vectors here. Real-gallery claims need measured persisted embeddings and documented provenance.
6. **Use:** research findings, not clinical validation or diagnostic recommendations.

For formulas, attribution, protocol, artifacts, and limitations, consult the canonical research documents linked above.
